In [0]:
DROP TABLE IF EXISTS workspace.silver.prd_info_silver;
CREATE OR REPLACE TABLE workspace.silver.prd_info_silver(
	prd_id INT,
	cat_id VARCHAR(50),
	prd_key VARCHAR(50),
	prd_nm VARCHAR(50),
	prd_cost INT,
	prd_line VARCHAR(50),
	prd_start_dt DATE,
	prd_end_dt DATE,
    dwh_create_date DATE default current_date()
) TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
DROP TABLE IF EXISTS workspace.silver.cust_info_silver;
CREATE OR REPLACE TABLE workspace.silver.cust_info_silver(
	cst_id INT,
	cst_key VARCHAR(50),
	cst_firstname VARCHAR(50),
	cst_lastname VARCHAR(50),
	cst_marital_status VARCHAR(50),
	cst_gndr VARCHAR(50),
	cst_create_date DATE,
    dwh_create_date DATE default current_date()
) TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
DROP TABLE IF EXISTS workspace.silver.sales_details_silver;
CREATE OR REPLACE TABLE workspace.silver.sales_details_silver(
    sls_ord_num VARCHAR(50),
    sls_prd_key VARCHAR(50),
    sls_cust_id INT,
    sls_order_dt DATE,
    sls_ship_dt DATE,
    sls_due_dt DATE,
    sls_sales INT,
    sls_quantity INT,
    sls_price INT,
    dwh_create_date DATE default current_date()
) TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
DROP TABLE IF EXISTS workspace.silver.CUST_AZ12_silver;
CREATE OR REPLACE TABLE workspace.silver.CUST_AZ12_silver(
    CID VARCHAR(50),
    BDATE DATE,
    GEN VARCHAR(50),
    dwh_create_date DATE default current_date()
) TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
DROP TABLE IF EXISTS workspace.silver.LOC_A101_silver;
CREATE OR REPLACE TABLE workspace.silver.LOC_A101_silver(
    CID VARCHAR(50),
    CNTRY VARCHAR(50),
    dwh_create_date DATE default current_date()
) TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
DROP TABLE IF EXISTS workspace.silver.PX_CAT_G1V2_silver;
CREATE OR REPLACE TABLE workspace.silver.PX_CAT_G1V2_silver(
    ID VARCHAR(50),
    CAT VARCHAR(50),
    SUBCAT VARCHAR(50),
    MAINTAINANCE VARCHAR(50),
    dwh_create_date DATE default current_date()
) TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
TRUNCATE TABLE workspace.silver.cust_info_silver;

INSERT INTO workspace.silver.cust_info_silver(cst_id, cst_key, cst_firstname, cst_lastname, cst_marital_status, cst_gndr, cst_create_date)
SELECT 
cst_id,
cst_key,
TRIM(cst_firstname) AS cst_firstname,
TRIM(cst_lastname) AS cst_lastname,
CASE WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
	WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
	ELSE 'n/a'
END cst_maritial_status,
CASE WHEN UPPER(TRIM(cst_gndr)) = 'M' THEN 'Male'
	WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
	ELSE 'n/a'
END cst_gndr,
cst_create_date
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last
		FROM workspace.bronze.cust_info_bronze
		WHERE cst_id IS NOT NULL)
t WHERE flag_last =  1

In [0]:
TRUNCATE TABLE workspace.silver.prd_info_silver;

INSERT INTO workspace.silver.prd_info_silver(prd_id, cat_id, prd_key, prd_nm, prd_cost, prd_line, prd_start_dt, prd_end_dt)
SELECT
	prd_id ,
	REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,
	SUBSTRING(prd_key, 7, len(prd_key)) AS prd_key,
	prd_nm,
	coalesce(prd_cost, 0) AS prd_cost,
	CASE WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
		WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
		WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'other Sales'
		WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Touring'
		ELSE 'n/a'
	END AS prd_line,
	CAST(prd_start_dt AS DATE) AS prd_start_dt,
	CAST(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY  prd_start_dt) -1 AS DATE) AS prd_end_dt
	FROM workspace.bronze.prd_info_bronze;


In [0]:
TRUNCATE TABLE workspace.silver.sales_details_silver;

INSERT INTO workspace.silver.sales_details_silver(sls_ord_num, sls_prd_key, sls_cust_id,sls_order_dt, sls_ship_dt, sls_due_dt, sls_sales, sls_quantity, sls_price)
SELECT
	sls_ord_num,
	sls_prd_key,
	sls_cust_id,
	CASE WHEN sls_order_dt = 0 OR LEN(sls_order_dt) != 8 THEN NULL
		ELSE to_date(CAST(sls_order_dt AS VARCHAR(10)), 'yyyyMMdd') 
	END AS sls_order_dt,
	CASE WHEN sls_ship_dt = 0 OR LEN(sls_ship_dt) != 8 THEN NULL
		ELSE to_date(CAST(sls_ship_dt AS VARCHAR(10)), 'yyyyMMdd') 
	END AS sls_ship_dt,
	CASE WHEN sls_due_dt = 0 OR LEN(sls_due_dt) != 8 THEN NULL
		ELSE to_date(CAST(sls_due_dt AS VARCHAR(10)), 'yyyyMMdd') 
	END AS sls_due_dt,
	CASE WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price)
			THEN sls_quantity * ABS(sls_price)
		ELSE sls_sales
	END AS sls_sales,
	sls_quantity,
	CASE WHEN sls_price IS NULL OR sls_price <= 0
			THEN sls_sales / sls_quantity
		ELSE  sls_price
	END AS sls_price
	FROM workspace.bronze.sales_details_bronze;

In [0]:
TRUNCATE TABLE workspace.silver.CUST_AZ12_silver;

INSERT INTO workspace.silver.CUST_AZ12_silver(CID, BDATE, GEN)
SELECT
	CASE WHEN CID LIKE 'NAS%' THEN SUBSTRING(CID, 4, LEN(CID))
		ELSE CID
	END AS CID,
	CASE WHEN BDATE > GETDATE() THEN NULL
		ELSE BDATE
	END AS BDATE,
	CASE 
		WHEN UPPER(TRIM(GEN)) IN ('M','MALE') THEN 'Male'
		WHEN UPPER(TRIM(GEN)) IN ('F','FEMALE') THEN 'Female'
		ELSE 'n/a'
	END AS GEN
FROM workspace.bronze.cust_az12_bronze;

In [0]:
TRUNCATE TABLE workspace.silver.LOC_A101_silver;

INSERT INTO workspace.silver.LOC_A101_silver(CID, CNTRY)
SELECT
REPLACE(CID, '-', '') AS CID,
CASE 
	WHEN CNTRY IN ('DE', 'Germany') THEN 'Germany'
	WHEN CNTRY IN ('USA', 'US', 'United States') THEN 'USA'
	WHEN CNTRY IS NULL OR TRIM(CNTRY) = ' ' THEN 'n/a'
	ELSE CNTRY
END AS CNTRY
FROM workspace.bronze.loc_a101_bronze;

In [0]:
TRUNCATE TABLE workspace.silver.PX_CAT_G1V2_silver;

INSERT INTO workspace.silver.PX_CAT_G1V2_silver(ID, CAT, SUBCAT, MAINTAINANCE)
SELECT
	ID,
	CAT,
	SUBCAT,
	MAINTENANCE
FROM workspace.bronze.px_cat_g1v2_bronze

In [0]:
SELECT * FROM workspace.silver.px_cat_g1v2_silver 